<div align="center">
  <img src="https://raw.githubusercontent.com/eightmm/FoldJAX/main/docs/banner-light.png" width="760" alt="FoldJAX">
  <h2>One input, multiple JAX structure models</h2>
  <p>Build one protein, nucleic-acid, or ligand job, run several FoldJAX models, and compare every result.</p>
</div>

### How to run

1. Choose **Runtime → Change runtime type → GPU or TPU** and a **Python 3.13** runtime. The experimental TPU path initially targets v5e.
2. Edit the form below. Each model is an independent checkbox; the default compares a two-chain insulin example with **Protenix + OpenDDE**.
3. Choose **Runtime → Run all**. The dependency cell restarts the runtime once; reconnect and choose **Run all** one more time.
4. Review the model table before the downloads begin. Prediction runs are deliberately sequential so several large checkpoints are not resident on the accelerator at once.

| Checkboxes | Accepted form input | Checkpoint setup |
|---|---|---|
| Protenix, OpenDDE, Boltz-2 | protein, DNA, RNA, CCD/SMILES ligands | public managed downloads |
| ESMFold2 | protein, DNA, RNA, ligands, modifications, covalent bonds | public complete bundle, about 26.77 GB |
| OpenFold3 | protein, DNA, RNA, CCD/SMILES ligands | public managed p1 checkpoint, about 2.29 GB |
| AlphaFold 3 | protein, DNA, RNA, CCD/SMILES ligands | manually obtained weight directory |

> **Accelerator scope:** NVIDIA is the validated Colab path. TPU support is experimental, initially targets v5e, and selects portable XLA kernels; a successful setup check does not replace per-model real-weight parity validation.

> **Scientific and privacy scope:** Fast demo mode uses 1 sample, 20 diffusion steps, and 1 recycle. It is a smoke/iteration schedule, not any model's released schedule. Native confidence fields are shown side by side but are not automatically ranked because names and calibration differ by model. MSA policy none does not contact an additional MSA service. Policy auto sends protein sequences to the public ColabFold MMseqs2 service. Colab, Google Drive, saved notebook outputs, and downloaded archives can still retain your input.


In [ ]:
# @title 1. Configure biomolecules and choose models
import re
from html import escape

from IPython.display import HTML, display


def foldjax_card(title, message, *, tone="info", metrics=()):
    palette = {
        "info": ("#2563eb", "#eff6ff"),
        "success": ("#059669", "#ecfdf5"),
        "warning": ("#d97706", "#fffbeb"),
        "error": ("#dc2626", "#fef2f2"),
        "neutral": ("#64748b", "#f8fafc"),
    }
    accent, background = palette[tone]
    chips = "".join(
        f"<span style='display:inline-block;background:white;border:1px solid "
        f"#dbe2ea;border-radius:999px;padding:5px 10px;margin:8px 6px 0 0;"
        f"font-size:12px;color:#334155'><b>{escape(str(label))}</b> "
        f"{escape(str(value))}</span>"
        for label, value in metrics
    )
    display(HTML(
        f"<div style='border:1px solid #dbe2ea;border-left:5px solid {accent};"
        f"border-radius:14px;padding:16px 18px;margin:12px 0;background:{background};"
        f"box-shadow:0 2px 8px rgba(15,23,42,.06)'>"
        f"<div style='font-size:16px;font-weight:700;color:#0f172a'>"
        f"{escape(str(title))}</div>"
        f"<div style='margin-top:5px;color:#475569;line-height:1.55'>"
        f"{escape(str(message))}</div><div>{chips}</div></div>"
    ))


def foldjax_table(frame, title, message=""):
    foldjax_card(title, message, tone="neutral")
    table_html = frame.to_html(index=False, escape=True, border=0)
    display(HTML(
        "<style>"
        ".foldjax-table table{border-collapse:separate;border-spacing:0;"
        "width:100%;font-size:13px}.foldjax-table th{background:#0f172a;"
        "color:white;text-align:left;padding:9px 10px;position:sticky;top:0}"
        ".foldjax-table td{padding:8px 10px;border-bottom:1px solid #e2e8f0;"
        "vertical-align:top}.foldjax-table tr:nth-child(even){background:#f8fafc}"
        "</style><div class='foldjax-table' style='overflow:auto;"
        "border:1px solid #e2e8f0;border-radius:12px;margin:0 0 18px'>"
        f"{table_html}</div>"
    ))

# @markdown **Input — use `:` between polymer chains**
JOB_NAME = "insulin-complex"  # @param {type:"string"}
PROTEIN_CHAINS = "GIVEQCCTSICSLYQLENYCN:FVNQHLCGSHLVEALYLVCGERGFFYTPKT"  # @param {type:"string"}  # noqa: E501
DNA_CHAINS = ""  # @param {type:"string"}
RNA_CHAINS = ""  # @param {type:"string"}
LIGAND_CCD_CODES = ""  # @param {type:"string"}
LIGAND_SMILES = ""  # @param {type:"string"}
# @markdown **Models — check any compatible combination**
RUN_PROTENIX = True  # @param {type:"boolean"}
RUN_OPENDDE = True  # @param {type:"boolean"}
RUN_BOLTZ2 = False  # @param {type:"boolean"}
RUN_ESMFOLD2 = False  # @param {type:"boolean"}
RUN_OPENFOLD3 = False  # @param {type:"boolean"}
RUN_ALPHAFOLD3 = False  # @param {type:"boolean"}
# @markdown **OpenFold3 p1 auto-downloads when blank.**
# @markdown **AlphaFold 3 parameters are required.**
OPENFOLD3_WEIGHTS_PATH = ""  # @param {type:"string"}
ALPHAFOLD3_WEIGHTS_PATH = ""  # @param {type:"string"}
# @markdown **Run policy**
RUN_MODE = "Fast demo"  # @param ["Fast demo", "Released defaults"]
NUM_SEEDS = 1  # @param {type:"integer", min:1, max:5}
MSA_POLICY = "none"  # @param ["none", "auto", "required"]
CONTINUE_ON_ERROR = True  # @param {type:"boolean"}
PERSIST_WEIGHTS_TO_DRIVE = False  # @param {type:"boolean"}
DOWNLOAD_RESULTS = True  # @param {type:"boolean"}

SELECTED_MODELS = tuple(
    model_name
    for model_name, enabled in (
        ("protenix", RUN_PROTENIX),
        ("opendde", RUN_OPENDDE),
        ("boltz2", RUN_BOLTZ2),
        ("esmfold2", RUN_ESMFOLD2),
        ("openfold3", RUN_OPENFOLD3),
        ("alphafold3", RUN_ALPHAFOLD3),
    )
    if enabled
)
if not SELECTED_MODELS:
    raise ValueError("Select at least one model checkbox.")
if not 1 <= int(NUM_SEEDS) <= 5:
    raise ValueError("NUM_SEEDS must be between 1 and 5.")

NUCLEIC_ALPHABETS = {
    "dna": frozenset("ACGTNRYKMSWBDHV"),
    "rna": frozenset("ACGUNRYKMSWBDHV"),
}

def split_polymer_chains(value, kind):
    compact = re.sub(r"\s+", "", value).upper()
    chains = tuple(part for part in compact.split(":") if part)
    if any(not chain.isalpha() for chain in chains):
        raise ValueError(
            "Polymer chains must contain letters and use ':' between chains."
        )
    allowed = NUCLEIC_ALPHABETS.get(kind)
    if allowed is not None:
        for chain_index, chain in enumerate(chains, start=1):
            invalid = next((base for base in chain if base not in allowed), None)
            if invalid is not None:
                raise ValueError(
                    f"{kind.upper()} chain {chain_index} contains unsupported "
                    f"base {invalid!r}; use IUPAC {''.join(sorted(allowed))}."
                )
    return chains

PROTEIN_SEQUENCES = split_polymer_chains(PROTEIN_CHAINS, "protein")
DNA_SEQUENCES = split_polymer_chains(DNA_CHAINS, "dna")
RNA_SEQUENCES = split_polymer_chains(RNA_CHAINS, "rna")
LIGAND_CCDS = tuple(
    part.strip().upper()
    for part in LIGAND_CCD_CODES.split(",")
    if part.strip()
)
LIGAND_SMILES_VALUES = tuple(
    part.strip() for part in LIGAND_SMILES.split(";") if part.strip()
)
ENTITY_COUNTS = {
    "protein": len(PROTEIN_SEQUENCES),
    "dna": len(DNA_SEQUENCES),
    "rna": len(RNA_SEQUENCES),
    "ligand": len(LIGAND_CCDS) + len(LIGAND_SMILES_VALUES),
}
INPUT_ENTITY_TYPES = frozenset(
    kind for kind, count in ENTITY_COUNTS.items() if count
)
if not INPUT_ENTITY_TYPES:
    raise ValueError("Add at least one polymer chain or ligand.")
INPUT_REQUIRED_FEATURES = frozenset(
    feature
    for feature, present in (
        ("ligand_ccd", bool(LIGAND_CCDS)),
        ("ligand_smiles", bool(LIGAND_SMILES_VALUES)),
    )
    if present
)
MANUAL_WEIGHT_TEXT = {
    "openfold3": OPENFOLD3_WEIGHTS_PATH.strip(),
    "alphafold3": ALPHAFOLD3_WEIGHTS_PATH.strip(),
}
JOB_SLUG = re.sub(r"[^a-z0-9._-]+", "-", JOB_NAME.lower()).strip("-")
if not JOB_SLUG:
    raise ValueError("JOB_NAME must contain at least one letter or digit.")

RUN_LABEL = "fast-demo" if RUN_MODE == "Fast demo" else "released"
FAST_SCHEDULE = (
    {"num_samples": 1, "num_steps": 20, "num_recycles": 1}
    if RUN_MODE == "Fast demo"
    else {}
)
entity_summary = ", ".join(
    f"{kind}={count}" for kind, count in ENTITY_COUNTS.items() if count
)
foldjax_card(
    "Configuration ready",
    "One common job will be validated independently by every selected model.",
    tone="success",
    metrics=(
        ("job", JOB_SLUG),
        ("entities", entity_summary),
        ("models", ", ".join(SELECTED_MODELS)),
        ("mode", RUN_MODE),
    ),
)


In [ ]:
# @title 2. Detect the accelerator and install its pinned FoldJAX stack
import os
import shutil
import signal
import subprocess
import sys
from pathlib import Path

FOLDJAX_REF = "8f3ebac133fb44ee1b7565573fdb5bea2faf1c8d"
JAX_VERSION = "0.11.1"

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"FoldJAX requires Python 3.13; this runtime has "
        f"{sys.version.split()[0]}. In Colab, choose Runtime → Change runtime "
        "type → Runtime version, select a Python 3.13 GPU or TPU runtime, "
        "reconnect, and rerun this notebook from the first cell. If Python "
        "3.13 is not offered, this notebook cannot run in that session."
    )

def detect_accelerator():
    tpu_environment = any(
        os.environ.get(name)
        for name in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_ACCELERATOR_TYPE")
    )
    if Path("/dev/accel0").exists() or tpu_environment:
        return "tpu"
    if shutil.which("nvidia-smi") is not None:
        gpu_probe = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        if gpu_probe.returncode == 0 and gpu_probe.stdout.strip():
            return "gpu"
    raise RuntimeError(
        "No supported accelerator was found. In Colab select Runtime → Change "
        "runtime type → GPU or TPU, reconnect, and run all cells again."
    )

ACCELERATOR_KIND = detect_accelerator()
install_extras = ["cuda12"] if ACCELERATOR_KIND == "gpu" else []
if "alphafold3" in SELECTED_MODELS:
    install_extras.append("alphafold3")
if "openfold3" in SELECTED_MODELS:
    install_extras.append("openfold3-preprocess")
install_profile = "-".join((ACCELERATOR_KIND, *install_extras))
extras_suffix = f"[{','.join(install_extras)}]" if install_extras else ""
foldjax_spec = (
    f"foldjax{extras_suffix} @ "
    f"git+https://github.com/eightmm/FoldJAX.git@{FOLDJAX_REF}"
)
accelerator_requirements = (
    [] if ACCELERATOR_KIND == "gpu" else [f"jax[tpu]=={JAX_VERSION}"]
)
install_marker = Path(
    f"/content/.foldjax-colab-{FOLDJAX_REF}-{install_profile}.installed"
)

if not install_marker.exists():
    foldjax_card(
        "Installing FoldJAX",
        f"Preparing the pinned Python 3.13 {ACCELERATOR_KIND.upper()} stack.",
        metrics=(("source", FOLDJAX_REF[:12]), ("profile", install_profile)),
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            foldjax_spec,
            *accelerator_requirements,
            "py3Dmol>=2.0,<3",
        ]
    )
    install_marker.write_text(FOLDJAX_REF)
    foldjax_card(
        "Dependencies installed",
        "The runtime will restart once. Reconnect and choose Run all again.",
        tone="success",
    )
    os.kill(os.getpid(), signal.SIGKILL)
else:
    foldjax_card(
        "Dependencies ready",
        "The pinned accelerator stack is already installed in this runtime.",
        tone="success",
        metrics=(("commit", FOLDJAX_REF[:12]), ("profile", install_profile)),
    )


<div style="border:1px solid #dbeafe;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#eff6ff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#2563eb">RUNTIME · STORAGE</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Fast local execution, optional persistent checkpoints</h3>
  <p style="margin:0;color:#475569">Choose where model assets live while keeping device-specific compilation local.</p>
</div>

Local Colab storage is fastest. Enable Drive persistence before **Run all** if you want downloaded and converted checkpoints to survive a runtime reset. Compilation entries stay local because they are tied to the JAX version, accelerator, options, weights, and concrete input shape.


In [ ]:
# @title 3. Configure storage and portable kernels
WORK_DIR = Path("/content/foldjax-colab")
OUTPUT_ROOT = WORK_DIR / "outputs" / JOB_SLUG / RUN_LABEL
COMPILE_CACHE = WORK_DIR / "compile-cache"

if PERSIST_WEIGHTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    foldjax_home = Path("/content/drive/MyDrive/foldjax-cache")
else:
    foldjax_home = Path("/content/foldjax-cache")

os.environ["FOLDJAX_HOME"] = str(foldjax_home)
if ACCELERATOR_KIND == "gpu":
    os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")
    triangle_multiplication_backend = "cueq"
    protenix_triangle_backend = "cueq_jit"
    KERNEL_PROFILE = "CUDA fused"
else:
    triangle_multiplication_backend = "xla"
    protenix_triangle_backend = "xla_jit"
    KERNEL_PROFILE = "portable XLA"
os.environ["PROTENIX_TRIANGLE_MULTIPLICATION_BACKEND"] = (
    triangle_multiplication_backend
)
os.environ["PROTENIX_TRIANGLE_BACKEND"] = protenix_triangle_backend
os.environ["BOLTZ_JAX_TRIANGLE_MULTIPLICATION_BACKEND"] = (
    triangle_multiplication_backend
)
for directory in (WORK_DIR, OUTPUT_ROOT, COMPILE_CACHE, foldjax_home):
    directory.mkdir(parents=True, exist_ok=True)

foldjax_card(
    "Storage ready",
    "Weights can persist in Drive; compilation cache and outputs stay local for speed.",
    tone="success",
    metrics=(
        ("weights", foldjax_home),
        ("outputs", OUTPUT_ROOT),
        ("compile cache", COMPILE_CACHE),
        ("storage", "Google Drive" if PERSIST_WEIGHTS_TO_DRIVE else "Colab local"),
        ("accelerator", ACCELERATOR_KIND.upper()),
        ("kernels", KERNEL_PROFILE),
    ),
)


In [ ]:
# @title 4. Verify Python, JAX, packages, and the selected accelerator
from importlib import metadata

import jax

expected_versions = {
    "cuequivariance": "0.11.1",
    "cuequivariance-jax": "0.11.1",
    "flax": "0.12.9",
    "jaxlib": "0.11.1",
    "qwix": "0.1.8",
    "tokamax": "0.0.13",
}
required_packages = set()
if ACCELERATOR_KIND == "gpu":
    expected_versions.update({
        "cuequivariance-ops-cu12": "0.11.1",
        "cuequivariance-ops-jax-cu12": "0.11.1",
        "jax-cuda12-pjrt": "0.11.1",
        "jax-cuda12-plugin": "0.11.1",
        "triton": "3.7.1",
    })
else:
    required_packages.add("libtpu")
installed_versions = {}
for package in expected_versions:
    try:
        installed_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_versions[package] = "missing"
version_mismatches = {
    package: (installed_versions[package], expected)
    for package, expected in expected_versions.items()
    if installed_versions[package] != expected
}
missing_packages = set()
for package in required_packages:
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        missing_packages.add(package)
if jax.__version__ != JAX_VERSION or version_mismatches or missing_packages:
    install_marker.unlink(missing_ok=True)
    raise RuntimeError(
        f"Expected JAX {JAX_VERSION} and the pinned accelerator stack, but got "
        f"JAX {jax.__version__} with mismatches {version_mismatches}. "
        f"Missing required packages: {sorted(missing_packages)}. "
        "Rerun the install cell; it will reinstall the stack and restart once."
    )

all_devices = jax.devices()
accelerator_devices = [
    device for device in all_devices if device.platform == ACCELERATOR_KIND
]
if not accelerator_devices:
    raise RuntimeError(
        f"No JAX {ACCELERATOR_KIND.upper()} was detected after installation. "
        "Reconnect, then run the notebook from the first cell."
    )
device_summary = str(accelerator_devices[0])
if ACCELERATOR_KIND == "gpu":
    gpu_info = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if gpu_info.stdout.strip():
        device_summary = gpu_info.stdout.strip().splitlines()[0]
try:
    memory_stats = accelerator_devices[0].memory_stats() or {}
except (AttributeError, RuntimeError, TypeError):
    memory_stats = {}
memory_limit = memory_stats.get("bytes_limit")
DEVICE_MEMORY_BYTES = memory_limit
memory_summary = (
    f"{memory_limit / 2**30:.1f} GiB HBM"
    if memory_limit is not None and ACCELERATOR_KIND == "tpu"
    else f"{memory_limit / 2**30:.1f} GiB VRAM"
    if memory_limit is not None
    else "capacity unavailable from this accelerator runtime"
)
foldjax_card(
    f"{ACCELERATOR_KIND.upper()} runtime verified",
    "Python, JAX, accelerator packages, and selected kernels are consistent.",
    tone="success",
    metrics=(
        ("Python", sys.version.split()[0]),
        ("JAX", jax.__version__),
        ("device", device_summary),
        ("memory", memory_summary),
        ("JAX devices", len(accelerator_devices)),
    ),
)


<div style="border:1px solid #ddd6fe;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#f5f3ff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#7c3aed">CHECKPOINTS · COMPATIBILITY</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Know what will download before setup begins</h3>
  <p style="margin:0;color:#475569">Selected models are checked against the common input, licence, disk, and runtime requirements.</p>
</div>

The next cell checks the selected entity types and ligand representations against every model before downloading anything. It shows publisher, licence, registered size, cache/runtime state, and compatibility constraints, then fetches public bundles **sequentially**. OpenFold3 uses the publisher's public p1 checkpoint; its path field is only an optional compatible-p1 override. AlphaFold3 still requires parameters you obtained under its publisher's terms. Re-running skips verified assets; FoldJAX never redistributes model weights.

The disk check is deliberately conservative because source archives and converted JAX files can coexist during first setup. ESMFold2's complete profile is particularly large; use a roomy runtime or Drive-backed store and begin with fewer checked models.


In [ ]:
# @title 5. Inspect and fetch the selected model weights
import pandas as pd

from foldjax import model_info

model_rows = []
model_infos = {}
MODEL_WEIGHTS = {}
setup_errors = []
pending_download_bytes = 0
for model_name in SELECTED_MODELS:
    info = model_info(model_name)
    model_infos[model_name] = info
    unsupported_types = INPUT_ENTITY_TYPES - set(info.capabilities.entity_types)
    unsupported_features = INPUT_REQUIRED_FEATURES - set(
        info.capabilities.common_schema_features
    )
    if unsupported_types or unsupported_features:
        setup_errors.append(
            f"{model_name} cannot represent this common input: "
            f"entity types={sorted(unsupported_types)}, "
            f"features={sorted(unsupported_features)}"
        )

    manual_text = MANUAL_WEIGHT_TEXT.get(model_name, "")
    manual_path = Path(manual_text).expanduser() if manual_text else None
    if manual_path is not None and not manual_path.exists():
        setup_errors.append(
            f"{model_name} manual weights do not exist: {manual_path}"
        )
    MODEL_WEIGHTS[model_name] = manual_path
    effective_ready = info.weights_ready or (
        manual_path is not None and manual_path.exists()
    )
    if not info.weights_fetchable and not effective_ready:
        setup_errors.append(
            f"{model_name} needs manually obtained weights. Set its path "
            f"in the first form, or place them at {info.weights_path}. "
            f"{info.notes}"
        )
    if (
        manual_path is None
        and not info.weights_ready
        and info.download_bytes is not None
    ):
        pending_download_bytes += info.download_bytes
    if model_name == "esmfold2":
        constraint = (
            "all-biomolecule NumPy adapter; complete structure, ESMC, "
            "and chemistry bundle ~26.77 GB"
        )
    elif model_name == "openfold3":
        constraint = "public managed p1 (~2.29 GB); p2/OpenBind incompatible"
    elif not info.weights_fetchable:
        constraint = "manual/gated parameters"
    else:
        constraint = "public managed bundle"
    model_rows.append(
        {
            "model": info.model,
            "weights": (
                "manual path"
                if manual_path is not None and manual_path.exists()
                else "ready" if info.weights_ready else "not ready"
            ),
            "runtime_ready": info.runtime.ready,
            "download_GB": (
                round(info.download_bytes / 1e9, 2)
                if info.download_bytes is not None
                else None
            ),
            "licence": info.weights_licence,
            "constraints": constraint,
            "source": info.weights_source,
        }
    )

foldjax_table(
    pd.DataFrame(model_rows),
    "Selected model compatibility",
    "Review licences, checkpoint size, and input constraints before setup.",
)
if setup_errors:
    raise RuntimeError("\n\n".join(setup_errors))
free_bytes = shutil.disk_usage(foldjax_home).free
required_bytes = max(
    8_000_000_000,
    int(pending_download_bytes * 1.8 + 5_000_000_000),
)
if free_bytes < required_bytes:
    raise RuntimeError(
        "There is not enough free storage for the selected uncached bundles "
        "and conversion headroom. Select fewer models or enable Drive storage."
    )
foldjax_card(
    "Storage check passed",
    "Enough room is available for source archives, conversion, and outputs.",
    tone="success",
    metrics=(
        ("free", f"{free_bytes / 1e9:.1f} GB"),
        ("pending downloads", f"{pending_download_bytes / 1e9:.1f} GB"),
        ("setup target", f"{required_bytes / 1e9:.1f} GB"),
    ),
)

for model_name in SELECTED_MODELS:
    info = model_infos[model_name]
    foldjax_card(
        f"Preparing {model_name}",
        "Checkpoint setup runs one model at a time to limit peak storage use.",
        metrics=(("source", info.weights_source),),
    )
    if (
        info.weights_fetchable
        and MODEL_WEIGHTS[model_name] is None
        and not info.weights_ready
    ):
        subprocess.run(
            [
                sys.executable,
                "-m",
                "foldjax.cli",
                "weights",
                "fetch",
                "--model",
                model_name,
            ],
            check=True,
        )
    elif MODEL_WEIGHTS[model_name] is not None:
        foldjax_card(
            f"Using supplied {model_name} parameters",
            "The compatible local checkpoint path is ready.",
            tone="neutral",
        )
    else:
        foldjax_card(
            f"Using cached {model_name} parameters",
            "The verified managed checkpoint is already ready.",
            tone="neutral",
        )
    if not info.runtime.ready:
        foldjax_card(
            f"Building the {model_name} runtime",
            "This one-time generated component is cached with the weights.",
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "foldjax.cli",
                "runtime",
                "prepare",
                "--model",
                model_name,
            ],
            check=True,
        )
foldjax_card(
    "All selected models are ready",
    "Checkpoint and generated-runtime setup completed successfully.",
    tone="success",
    metrics=(("models", ", ".join(SELECTED_MODELS)), ("store", foldjax_home)),
)


<div style="border:1px solid #bbf7d0;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#ecfdf5,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#059669">ONE INPUT · MANY MODELS</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Record the experiment once</h3>
  <p style="margin:0;color:#475569">Protein, nucleic acid, and ligand entities become one reusable common-schema job.</p>
</div>

Protein, DNA, and RNA chains each use a colon as the chain separator. Repeating a sequence creates a homomer. Ligand CCD codes use commas; separate multiple SMILES ligands with semicolons. FoldJAX assigns chain IDs in form order, writes one common-schema job, and sends that exact recorded input to every compatible selected backend.

With MSA policy none, no additional MSA service is contacted. With auto or required, protein sequences are sent to the public ColabFold MMseqs2 service and the reusable alignment cache is shared across model runs; public MMseqs2 does not serve RNA. The notebook prints entity counts, not sequences or SMILES, so a saved output cell does not echo private input text. OpenFold3's compatible p1 checkpoint is a public managed download; AlphaFold3 still requires parameters obtained separately under its publisher's terms. ESMFold2 accepts protein, DNA, RNA, CCD/SMILES ligands, modifications, and covalent bonds; its complete public structure+ESMC+chemistry bundle is about 26.77 GB.


In [ ]:
# @title 6. Build the common FoldJAX job
from foldjax import Job

job = Job.from_sequences(
    protein=PROTEIN_SEQUENCES,
    dna=DNA_SEQUENCES,
    rna=RNA_SEQUENCES,
    ligand_ccd=LIGAND_CCDS,
    ligand_smiles=LIGAND_SMILES_VALUES,
    name=JOB_SLUG,
)
job_path = job.write(WORK_DIR / "input" / f"{JOB_SLUG}.json")
foldjax_card(
    "Common job recorded",
    "Every selected backend will read this same normalized input file.",
    tone="success",
    metrics=(
        ("job", JOB_SLUG),
        ("entities", entity_summary),
        ("input file", job_path),
    ),
)


<div style="border:1px solid #fed7aa;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#fff7ed,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#ea580c">EXECUTION · SEQUENTIAL ACCELERATOR SESSIONS</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Review the plan, then run safely</h3>
  <p style="margin:0;color:#475569">Each backend gets the same recorded job while retaining its own validated runtime options.</p>
</div>

Every selected model receives the same common job path, seed policy, MSA policy, and sampling mode. Requests remain model-specific because their safe execution knobs are genuinely different: the first three use portable float32/XLA, OpenFold3 selects XLA triangles, AlphaFold3 selects XLA attention, and ESMFold2 keeps its own runtime contract. FoldJAX releases each backend session before opening the next. Resume mode validates finished manifests and avoids repeating completed work.

If **Continue on error** is enabled, an out-of-memory or model-specific setup failure is recorded while later models still run. Programming defects and user cancellation still stop immediately.


In [ ]:
# @title 7. Resolve and review every model run
from foldjax import PredictionRequest, resolve_requests

triangle_kernel = "cueq" if ACCELERATOR_KIND == "gpu" else "xla"
alphafold_attention = "auto" if ACCELERATOR_KIND == "gpu" else "xla"
MODEL_PRECISION = {
    "protenix": "native bfloat16",
    "opendde": "float32",
    "boltz2": "native bfloat16",
    "esmfold2": "native mixed/bfloat16",
    "openfold3": "float32",
    "alphafold3": "native bfloat16",
}
MODEL_OPTIONS = {
    "protenix": {
        "attention_kernel": "xla",
        "triangle_kernel": triangle_kernel,
    },
    "opendde": {"dtype": "float32", "attention_kernel": "xla"},
    "boltz2": {
        "attention_kernel": "xla",
        "triangle_kernel": triangle_kernel,
    },
    "esmfold2": {},
    "openfold3": {"triangle_kernel": triangle_kernel},
    "alphafold3": {"attention_kernel": alphafold_attention},
}
model_requests = tuple(
    PredictionRequest(
        model=model_name,
        input=job_path,
        weights=MODEL_WEIGHTS[model_name],
        output_dir=OUTPUT_ROOT / model_name / JOB_SLUG,
        cache_dir=COMPILE_CACHE,
        seed=0,
        num_seeds=int(NUM_SEEDS),
        msa=MSA_POLICY,
        options=MODEL_OPTIONS[model_name],
        resume=True,
        on_error="continue" if CONTINUE_ON_ERROR else "stop",
        **FAST_SCHEDULE,
    )
    for model_name in SELECTED_MODELS
)
resolved_runs = tuple(
    resolve_requests(model_request)[0] for model_request in model_requests
)
plan_rows = [
    {
        "model": run.model,
        "input": Path(run.input).name,
        "output": str(run.output_dir),
        "seeds": list(run.resolved_seeds),
        "options": run.options or "model runtime defaults",
        "precision": MODEL_PRECISION[run.model],
        "schedule": run.sampling or "released defaults",
    }
    for run in resolved_runs
]
foldjax_table(
    pd.DataFrame(plan_rows),
    "Resolved execution plan",
    "Each row is independently validated but uses the same common input.",
)


In [ ]:
# @title 8. Run all selected models
import json

from foldjax import BatchReport, predict_batch, progress

progress.enable()
partial_reports = []
for model_request in model_requests:
    foldjax_card(
        f"Running {model_request.model}",
        "The accelerator session opens for this model and closes before the next.",
        metrics=(("seeds", model_request.num_seeds), ("MSA", model_request.msa)),
    )
    partial_reports.append(predict_batch(model_request))
report = BatchReport(
    results=tuple(
        result for partial in partial_reports for result in partial.results
    ),
    failures=tuple(
        failure for partial in partial_reports for failure in partial.failures
    ),
    skipped=tuple(
        path for partial in partial_reports for path in partial.skipped
    ),
)
foldjax_card(
    "Prediction batch complete",
    "Successful structures are ready for validation and side-by-side analysis.",
    tone="warning" if report.failures else "success",
    metrics=(
        ("results", len(report.results)),
        ("failures", len(report.failures)),
        ("resumed", len(report.skipped)),
    ),
)
if report.failures:
    foldjax_table(
        pd.DataFrame(failure.summary() for failure in report.failures),
        "Model failures",
        "These runs failed; later models continued because Continue on error "
        "is enabled.",
    )


<div style="border:1px solid #bae6fd;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#f0f9ff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#0284c7">ANALYSIS · NATIVE SCORES</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Compare evidence, not a fabricated leaderboard</h3>
  <p style="margin:0;color:#475569">Structures are parsed and scores validated before side-by-side inspection.</p>
</div>

The table keeps each model's native score names. A similarly named metric can still differ in calibration, input representation, and sampling schedule, so the notebook does not combine them into a universal ranking. Use the table to inspect consistency and uncertainty, and use the interactive viewer for structural comparison.


In [ ]:
# @title 9. Validate structures and build the comparison table
import math

import gemmi

comparison_rows = []
STRUCTURES = {}
for result in report.results:
    for sample_index, sample in enumerate(result.samples, start=1):
        if sample.structure_path is None:
            raise RuntimeError(f"{result.model} returned no structure path.")
        structure_path = Path(sample.structure_path)
        if not structure_path.is_file() or structure_path.stat().st_size == 0:
            raise RuntimeError(f"Missing or empty structure: {structure_path}")
        if structure_path.suffix.lower() in {".cif", ".mmcif"}:
            gemmi.cif.read_file(str(structure_path))
        else:
            gemmi.read_structure(str(structure_path))
        if not sample.scores or not all(
            math.isfinite(float(value)) for value in sample.scores.values()
        ):
            raise RuntimeError(
                f"{result.model} returned invalid confidence scores: "
                f"{sample.scores}"
            )

        label = f"{result.model} · seed {sample.seed} · sample {sample_index}"
        STRUCTURES[label] = structure_path
        row = {
            "model": result.model,
            "seed": sample.seed,
            "sample": sample_index,
            "structure": structure_path.name,
        }
        row.update(
            {
                f"score:{name}": float(value)
                for name, value in sorted(sample.scores.items())
            }
        )
        comparison_rows.append(row)

if not comparison_rows:
    raise RuntimeError(
        "No prediction produced a usable structure. Review the failure table above."
    )
comparison = pd.DataFrame(comparison_rows).sort_values(
    ["model", "seed", "sample"],
    ignore_index=True,
)
foldjax_table(
    comparison.fillna("—"),
    "Validated prediction comparison",
    "Native confidence fields stay model-specific; no artificial global rank is added.",
)
foldjax_card(
    "Analysis ready",
    "All listed structure files parsed successfully and every score is finite.",
    tone="success",
    metrics=(
        ("models", comparison["model"].nunique()),
        ("structures", len(STRUCTURES)),
    ),
)


<div style="border:1px solid #a7f3d0;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#ecfeff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#0f766e">VISUALIZATION · STRUCTURE EXPLORER</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Inspect every model, seed, and sample interactively</h3>
  <p style="margin:0;color:#475569">Switch predictions and coloring without rerunning any model.</p>
</div>

Select any model/sample result without rerunning prediction. Chain spectrum is best for complexes; confidence coloring reads the structure's B-factor field when the writer stores per-residue confidence there.


In [ ]:
# @title 10. Interactive model and color selector
import ipywidgets as widgets
import py3Dmol
from IPython.display import clear_output

prediction_selector = widgets.Dropdown(
    options=list(STRUCTURES),
    description="Prediction:",
    layout=widgets.Layout(width="70%"),
)
color_selector = widgets.Dropdown(
    options=("Chain spectrum", "Confidence (B-factor)"),
    description="Color:",
    layout=widgets.Layout(width="45%"),
)
viewer_output = widgets.Output()


def render_structure(*_changes):
    structure_path = STRUCTURES[prediction_selector.value]
    structure_format = (
        "cif"
        if structure_path.suffix.lower() in {".cif", ".mmcif"}
        else "pdb"
    )
    with viewer_output:
        clear_output(wait=True)
        viewer = py3Dmol.view(width=900, height=560)
        viewer.addModel(structure_path.read_text(), structure_format)
        if color_selector.value == "Confidence (B-factor)":
            viewer.setStyle(
                {
                    "cartoon": {
                        "colorscheme": {
                            "prop": "b",
                            "gradient": "roygb",
                            "min": 0,
                            "max": 100,
                        }
                    }
                }
            )
        else:
            viewer.setStyle({"cartoon": {"color": "spectrum"}})
        viewer.setBackgroundColor("white")
        viewer.zoomTo()
        viewer.show()


foldjax_card(
    "Structure explorer ready",
    "Switch model, seed, sample, and coloring without rerunning prediction.",
    tone="success",
    metrics=(("predictions", len(STRUCTURES)), ("viewer", "py3Dmol")),
)
prediction_selector.observe(render_structure, names="value")
color_selector.observe(render_structure, names="value")
display(widgets.VBox([prediction_selector, color_selector, viewer_output]))
render_structure()


<div style="border:1px solid #cbd5e1;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#f8fafc,#f1f5f9)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#475569">EXPORT · REPRODUCIBLE BUNDLE</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Keep the input, analysis, and artifacts together</h3>
  <p style="margin:0;color:#475569">Create one portable archive for review, sharing, or later reproduction.</p>
</div>

The archive contains the common input job, a CSV comparison table, the full batch report, and every selected model's output directory, including structures, confidence artifacts, and FoldJAX run manifests.


In [ ]:
# @title 11. Package all model outputs
import zipfile

comparison_path = WORK_DIR / f"{JOB_SLUG}-comparison.csv"
report_path = WORK_DIR / f"{JOB_SLUG}-batch-report.json"
archive_path = WORK_DIR / f"{JOB_SLUG}-foldjax-comparison.zip"
comparison.to_csv(comparison_path, index=False)
report_path.write_text(json.dumps(report.summary(), indent=2))

artifact_count = 0
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(job_path, "input/job.json")
    bundle.write(comparison_path, "comparison.csv")
    bundle.write(report_path, "batch_report.json")
    for artifact in sorted(OUTPUT_ROOT.rglob("*")):
        if artifact.is_file():
            artifact_count += 1
            bundle.write(
                artifact,
                Path("outputs") / artifact.relative_to(OUTPUT_ROOT),
            )

foldjax_card(
    "Comparison archive ready",
    "The bundle contains the common job, validation table, report, and "
    "model artifacts.",
    tone="success",
    metrics=(
        ("file", archive_path.name),
        ("size", f"{archive_path.stat().st_size / 1e6:.1f} MB"),
        ("artifacts", artifact_count + 3),
    ),
)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files

        files.download(str(archive_path))
    except ImportError:
        foldjax_card(
            "Automatic download unavailable",
            "Download the archive from the notebook file browser.",
            tone="warning",
        )


<div style="border:1px solid #e2e8f0;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#ffffff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#64748b">NEXT · FROM DEMO TO PRODUCTION</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Scale up deliberately after the workflow succeeds</h3>
  <p style="margin:0;color:#475569">Move from the fast smoke schedule to released defaults, more seeds, and richer inputs.</p>
</div>

- Switch to **Released defaults** only after the fast workflow succeeds. The public models have different native sample/recycle defaults and released mode can take substantially longer.
- Add more seeds to inspect within-model variability. FoldJAX keeps every seed in the same reproducible result tree.
- Start with Protenix only if storage is tight. OpenDDE and Boltz-2 add public all-atom comparisons. ESMFold2 accepts the same all-biomolecule common job, but its complete structure+ESMC+chemistry bundle is about 26.77 GB.
- OpenFold3 downloads the public p1 checkpoint when its path is blank; a mounted Drive file can override it only with a compatible p1 checkpoint. Upstream p2/OpenBind checkpoints are not compatible with this port. AlphaFold3 still needs a supplied parameter directory.
- Re-running the prediction cell uses resume manifests. Re-running with a different input, model list, shape, options, weights, or runtime identity gets separate validated work.
- The form covers protein, DNA, RNA, CCD ligands, and SMILES ligands. For templates, modifications, covalent bonds, affinity, representations, or reusable job files, see the [FoldJAX README](https://github.com/eightmm/FoldJAX).
